# Outlier theo thống kê hay theo nghiệp vụ: những ngày lạ của Capital Bikeshare

Bộ `bike-sharing-uci`: `day.csv` (731 ngày, 2011–2012) và `hour.csv` (17 379 giờ) lượt thuê xe đạp ở Washington DC,
kèm thời tiết (`weathersit` 1 = quang, 3 = mưa/tuyết nhẹ), độ ẩm `hum` và tốc độ gió `windspeed` đã chuẩn hoá về 0–1.

Trưởng nhóm dự báo nhu cầu nhắn: **"Trước khi train model dự báo lượt thuê theo ngày, bỏ outlier giúp anh nhé."**
Bạn có thể chạy IQR trên cột `cnt` và gửi lại trong 5 phút. Bài này cho thấy vì sao làm thế là sai: quy tắc thống
kê không thấy những ngày lạ thật, lại đánh dấu những giờ hoàn toàn bình thường. Outlier là câu hỏi **nghiệp vụ**
("ngày này có đại diện cho nhu cầu bình thường không?"), không phải câu hỏi về khoảng cách tới trung bình.

Việc của bạn: viết ba cách phát hiện (IQR, z-score, tỉ lệ so với trung vị trượt), đếm giờ có dữ liệu mỗi ngày, gộp
thành bảng cờ theo ngày, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/basic/03-bike-outliers -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv", parse_dates=["dteday"])
hour = pd.read_csv(BIKE / "hour.csv", parse_dates=["dteday"])
print(day.shape, hour.shape, f"thiếu {len(day) * 24 - len(hour)} giờ so với 731 × 24")
day[["cnt", "hum", "windspeed"]].describe().round(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Hai quy tắc thống kê

- `iqr_flags(s, k=1.5) -> Series[bool]`: `True` khi giá trị nằm ngoài `[Q1 - k·IQR, Q3 + k·IQR]`, với `Q1`, `Q3`
  là `s.quantile(0.25)`, `s.quantile(0.75)` (nội suy tuyến tính, mặc định của pandas), `IQR = Q3 - Q1`.
- `zscore_flags(s, z=3.0) -> Series[bool]`: `True` khi `|s - mean| / std > z` (`std` mặc định của pandas, `ddof=1`).

Cả hai giữ index của `s`. Lưu ý hiệu ứng **masking** của z-score: một điểm cực lớn làm `std` phình ra, điểm lớn thứ
hai không còn vượt ngưỡng.

In [ ]:
def iqr_flags(s: pd.Series, k: float = 1.5) -> pd.Series:
    """True khi s nằm ngoài [Q1 - k*IQR, Q3 + k*IQR]."""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)


def zscore_flags(s: pd.Series, z: float = 3.0) -> pd.Series:
    """True khi |s - mean| / std > z."""
    return ((s - s.mean()) / s.std()).abs() > z

In [ ]:
# explore: IQR và z-score trên lượt thuê theo NGÀY và theo GIỜ
print("ngày bị IQR đánh dấu:", int(iqr_flags(day["cnt"]).sum()), "| z-score:", int(zscore_flags(day["cnt"]).sum()))
low = day.nsmallest(1, "cnt").iloc[0]
print(f"ngày thấp nhất {low.dteday:%Y-%m-%d}: {low.cnt} lượt, z = {(low.cnt - day.cnt.mean()) / day.cnt.std():.2f}")

h_flag = iqr_flags(hour["cnt"])
print(f"giờ bị IQR đánh dấu: {h_flag.sum()} / {len(hour)}")
print(hour.loc[h_flag, "hr"].value_counts().head(6).to_dict())
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(hour["hr"] + np.random.default_rng(0).uniform(-0.3, 0.3, len(hour)), hour["cnt"],
           s=2, alpha=0.15, c=np.where(h_flag, "#e45756", "#9aa5b1"))
ax.set_xlabel("giờ trong ngày")
ax.set_ylabel("lượt thuê/giờ")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Điểm đỏ = 'outlier' theo IQR: gần như toàn bộ là giờ cao điểm đi làm", loc="left")
plt.show()

## 2. Quy tắc nghiệp vụ: so với ngày "bình thường" quanh nó

Nhu cầu thay đổi theo mùa (mùa đông ít hơn mùa hè gấp đôi), nên 800 lượt là bình thường vào tháng 1 nhưng bất thường
vào tháng 7. So mỗi ngày với **trung vị trượt** của các ngày quanh nó:

`demand_ratio(day, window=29) -> Series`: `cnt` chia cho trung vị trượt **giữa** (`center=True`) cửa sổ `window`
ngày, `min_periods = window // 2 + 1` (ngày đầu và cuối năm vẫn có giá trị). Index là `dteday`.

Trung vị (không phải trung bình) để chính ngày lạ không kéo mốc so sánh về phía nó.

In [ ]:
def demand_ratio(day: pd.DataFrame, window: int = 29) -> pd.Series:
    """cnt / trung vị trượt giữa `window` ngày, index dteday."""
    cnt = day.set_index("dteday")["cnt"]
    return cnt / cnt.rolling(window, center=True, min_periods=window // 2 + 1).median()

## 3. Độ phủ: ngày đó có đủ 24 giờ dữ liệu không?

Trước khi gọi một con số là "bất thường", kiểm nó được đo trên bao nhiêu dữ liệu. `hour.csv` thiếu 165 giờ: có giờ
không ai thuê, có giờ hệ thống ngừng. Một ngày chỉ có 1 giờ dữ liệu thì tổng thấp là chuyện đương nhiên.

`hours_recorded(hour) -> Series`: số dòng (giờ có dữ liệu) mỗi `dteday`.

In [ ]:
def hours_recorded(hour: pd.DataFrame) -> pd.Series:
    """Số giờ có dữ liệu mỗi ngày, index dteday."""
    return hour.groupby("dteday").size()

## 4. Bảng cờ theo ngày

`flag_days(day, hour, window=29) -> DataFrame`, index `dteday`, đúng các cột theo thứ tự:

| Cột | Ý nghĩa |
| --- | --- |
| `cnt` | lượt thuê cả ngày |
| `hours` | số giờ có dữ liệu (`hours_recorded`; ngày không có giờ nào là 0) |
| `iqr` | `iqr_flags(cnt)` |
| `zscore` | `zscore_flags(cnt)` |
| `demand_drop` | `demand_ratio(day, window) < 0.4` |
| `sensor_error` | `hum == 0`: độ ẩm 0% ngoài trời ở DC không có thật, đây là lỗi cảm biến |

In [ ]:
def flag_days(day: pd.DataFrame, hour: pd.DataFrame, window: int = 29) -> pd.DataFrame:
    """Bảng cờ outlier theo ngày: cnt, hours, iqr, zscore, demand_drop, sensor_error."""
    d = day.set_index("dteday")
    return pd.DataFrame({
        "cnt": d["cnt"],
        "hours": hours_recorded(hour).reindex(d.index, fill_value=0),
        "iqr": iqr_flags(d["cnt"]),
        "zscore": zscore_flags(d["cnt"]),
        "demand_drop": demand_ratio(day, window) < 0.4,
        "sensor_error": d["hum"] == 0,
    })

In [ ]:
# explore: những ngày bị đánh dấu, kèm thời tiết, ngày lễ, độ phủ
flags = flag_days(day, hour)
print(flags[["iqr", "zscore", "demand_drop", "sensor_error"]].sum().to_dict())
odd = flags[flags.demand_drop | flags.sensor_error].join(
    day.set_index("dteday")[["weathersit", "holiday", "workingday", "hum"]]).join(demand_ratio(day).rename("ratio"))
odd.round(3)

In [ ]:
# explore: lượt thuê theo ngày, trung vị trượt, và ngày bị đánh dấu
d = day.set_index("dteday")["cnt"]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(d.index, d, color="#9aa5b1", lw=0.8, label="lượt thuê/ngày")
ax.plot(d.index, d.rolling(29, center=True, min_periods=15).median(), color="#4c78a8", label="trung vị trượt 29 ngày")
drop = flags[flags.demand_drop]
ax.scatter(drop.index, drop.cnt, color="#e45756", zorder=3, label="demand_drop (< 40% trung vị)")
sandy = pd.Timestamp("2012-10-29")
ax.annotate(f"29/10/2012: {flags.loc[sandy, 'cnt']} lượt, chỉ {flags.loc[sandy, 'hours']} giờ có dữ liệu",
            (sandy, flags.loc[sandy, "cnt"]), xytext=(-260, 10), textcoords="offset points", fontsize=8,
            arrowprops={"arrowstyle": "->", "color": "#555"})
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("IQR và z-score không đánh dấu ngày nào; so với trung vị trượt thấy 17 ngày lạ", loc="left")
plt.show()

## 5. Quyết định cho từng loại, rồi làm trắc nghiệm

Với mỗi ngày bị đánh dấu, quyết định theo **mục đích** (model dự báo nhu cầu ngày thường):

1. **Lỗi đo** (`sensor_error`): giá trị sai, nhưng lượt thuê ngày đó vẫn đúng. Sửa cột sai (đặt thiếu rồi xử lý như
   dữ liệu thiếu), không bỏ cả ngày.
2. **Sự kiện thật** (bão, tuyết, Giáng sinh): nhu cầu thật sự thấp. Báo cáo doanh thu tháng giữ nguyên; model dự báo
   ngày thường có thể loại hoặc thêm biến thời tiết/ngày lễ. Ghi rõ quyết định.
3. **Thiếu dữ liệu** (ít giờ có dữ liệu): con số thấp vì đo thiếu, không phải nhu cầu thấp. Không dùng như một quan
   sát nhu cầu.
4. Giờ cao điểm bị IQR đánh dấu: **không** phải outlier, đó là tín hiệu chính model cần học.

Xong thì mở app, làm phần **trắc nghiệm** của bài.